In [1]:
import math
import numpy as np
import pandas as pd
from scipy.stats import chi2, binom
from scipy.special import xlogy

In [2]:
# Step 1: Configuration

In [3]:
MODEL_FILES = {
    'Historical': (r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\historical_simulation_var_es.csv', 'VaR_1d_$_'),
    'EWMA':       (r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\ewma_scaled_var_es.csv',           'EWMA_VaR_1d_$_'),
    'GARCH':      (r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\garch_scaled_var_es.csv',          'GARCH_VaR_1d_$_'),
    'GJR-GARCH':  (r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\gjr_garch_scaled_var_es.csv',      'GJR_VaR_1d_$_'),
}
windows = [250, 500, 750]
confidence_levels = [0.95, 0.99, 0.995]
 
SIGNIFICANCE = 0.05   # a model "passes" if its p-values are at or above this
BASEL_WINDOW = 250    # Basel's traffic-light test is defined on 250 trading days
RUN_SELF_TESTS = True

In [4]:
#  Step 2: Statistical tests

In [6]:
def kupiec_pof(x, n, p):
    if n == 0:
        return np.nan, np.nan
    pi_hat = x / n
    ll_null = xlogy(n - x, 1 - p) + xlogy(x, p)
    ll_alt = xlogy(n - x, 1 - pi_hat) + xlogy(x, pi_hat)
    lr = max(-2.0 * (ll_null - ll_alt), 0.0)       # guard against tiny negative float error
    return lr, chi2.sf(lr, df=1)
 
 
def christoffersen_independence(hits):
    h = np.asarray(hits, dtype=int)
    prev, curr = h[:-1], h[1:]
    n00 = int(np.sum((prev == 0) & (curr == 0)))
    n01 = int(np.sum((prev == 0) & (curr == 1)))
    n10 = int(np.sum((prev == 1) & (curr == 0)))
    n11 = int(np.sum((prev == 1) & (curr == 1)))
    total = n00 + n01 + n10 + n11
    if h.sum() == 0 or total == 0:
        return np.nan, np.nan
    pi01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / total
    ll_null = xlogy(n00 + n10, 1 - pi) + xlogy(n01 + n11, pi)
    ll_alt = (xlogy(n00, 1 - pi01) + xlogy(n01, pi01)
              + xlogy(n10, 1 - pi11) + xlogy(n11, pi11))
    lr = max(-2.0 * (ll_null - ll_alt), 0.0)
    return lr, chi2.sf(lr, df=1)
 
 
def conditional_coverage(lr_pof, lr_ind):
    if np.isnan(lr_ind):
        return np.nan, np.nan
    lr = lr_pof + lr_ind
    return lr, chi2.sf(lr, df=2)
 
 
def basel_zone(exceptions_in_250_days):
    """Basel traffic light for a 99% 1-day VaR over 250 trading days."""
    if exceptions_in_250_days <= 4:
        return 'Green'
    if exceptions_in_250_days <= 9:
        return 'Yellow'
    return 'Red'

In [7]:
# Step 3: Self-test

In [8]:
def run_self_tests():
    print("\nRunning self-tests on the test functions themselves...")
 
    # (a) Kupiec against an independent formula built from binomial log-probabilities
    for x, n, p in [(5, 250, 0.01), (12, 750, 0.01), (0, 750, 0.005), (40, 750, 0.05)]:
        lr_ref = -2.0 * (binom.logpmf(x, n, p) - binom.logpmf(x, n, x / n))
        lr, pval = kupiec_pof(x, n, p)
        assert abs(lr - lr_ref) < 1e-9, f"Kupiec mismatch at x={x}, n={n}, p={p}"
        assert abs(pval - chi2.sf(lr_ref, 1)) < 1e-9
    print("  [ok] Kupiec matches an independent binomial-likelihood calculation")
 
    # (b) Christoffersen against a plain-Python, loop-based calculation
    seq = [0,0,1,1,0,0,0,1,0,0,1,1,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,1]
    c = {'00': 0, '01': 0, '10': 0, '11': 0}
    for a, b in zip(seq[:-1], seq[1:]):
        c[f"{a}{b}"] += 1
    xl = lambda k, q: k * math.log(q) if k > 0 else 0.0
    pi01 = c['01'] / (c['00'] + c['01']); pi11 = c['11'] / (c['10'] + c['11'])
    pi = (c['01'] + c['11']) / sum(c.values())
    ref = -2 * (xl(c['00'] + c['10'], 1 - pi) + xl(c['01'] + c['11'], pi)
                - xl(c['00'], 1 - pi01) - xl(c['01'], pi01)
                - xl(c['10'], 1 - pi11) - xl(c['11'], pi11))
    lr, _ = christoffersen_independence(seq)
    assert abs(lr - ref) < 1e-9, "Christoffersen independence mismatch"
    lr_k, _ = kupiec_pof(sum(seq), len(seq), 0.05)
    lr_cc, p_cc = conditional_coverage(lr_k, lr)
    assert abs(lr_cc - (lr_k + lr)) < 1e-12 and abs(p_cc - chi2.sf(lr_cc, 2)) < 1e-12
    print("  [ok] Christoffersen independence and conditional coverage match a hand-built calculation")
 
    # (c) The Basel table must equal the cumulative-binomial rule it was built from
    for x in range(0, 21):
        cum = binom.cdf(x, 250, 0.01)
        expected = 'Green' if cum < 0.95 else ('Yellow' if cum < 0.9999 else 'Red')
        assert basel_zone(x) == expected, f"Basel zone mismatch at {x} exceptions"
    print("  [ok] Basel zones (0-4 / 5-9 / 10+) match the cumulative-binomial thresholds")
 
    # (d) Behavior check: right size on i.i.d. data, real power against clustered data
    rng = np.random.default_rng(2024)
    reps, n, p = 2000, 750, 0.05
    iid = rng.random((reps, n)) < p
    size_k = np.mean([kupiec_pof(r.sum(), n, p)[1] < 0.05 for r in iid])
    size_i = np.mean([christoffersen_independence(r)[1] < 0.05 for r in iid])
    assert 0.03 <= size_k <= 0.08 and 0.03 <= size_i <= 0.08, "Tests are mis-sized on i.i.d. data"
 
    pi01_s, pi11_s = 0.03, 0.40                       # clustered: P(hit | hit yesterday) = 40%
    cl = np.zeros((reps, n), dtype=bool)
    cl[:, 0] = rng.random(reps) < 0.0476
    u = rng.random((reps, n))
    for t in range(1, n):
        cl[:, t] = u[:, t] < np.where(cl[:, t - 1], pi11_s, pi01_s)
    power_i = np.mean([christoffersen_independence(r)[1] < 0.05 for r in cl])
    kup_on_cl = np.mean([kupiec_pof(r.sum(), n, p)[1] < 0.05 for r in cl])
    assert power_i >= 0.70, "Independence test has too little power against clustered exceptions"
    print(f"  [ok] i.i.d. exceptions: Kupiec rejects {size_k:.1%}, Independence rejects {size_i:.1%} (target ~5%)")
    print(f"  [ok] clustered exceptions: Independence rejects {power_i:.1%}, "
          f"Kupiec only {kup_on_cl:.1%} - Kupiec cannot see clustering, which is why both tests exist")
    print("All self-tests passed.\n")
 
 
if RUN_SELF_TESTS:
    run_self_tests()


Running self-tests on the test functions themselves...
  [ok] Kupiec matches an independent binomial-likelihood calculation
  [ok] Christoffersen independence and conditional coverage match a hand-built calculation
  [ok] Basel zones (0-4 / 5-9 / 10+) match the cumulative-binomial thresholds
  [ok] i.i.d. exceptions: Kupiec rejects 4.5%, Independence rejects 6.9% (target ~5%)
  [ok] clustered exceptions: Independence rejects 99.9%, Kupiec only 19.9% - Kupiec cannot see clustering, which is why both tests exist
All self-tests passed.



In [9]:
# Step 4: Loading the four models and aligning them to ONE common test period 

In [10]:
print("Loading model outputs from Phases 4-7...")
model_data = {name: pd.read_csv(f, index_col=0, parse_dates=True) for name, (f, _) in MODEL_FILES.items()}
 
for name, df in model_data.items():
    print(f"  {name:<11} {len(df)} out-of-sample days  ({df.index.min().date()} to {df.index.max().date()})")
 
# The models do NOT start on exactly the same day (the EWMA series loses one extra
# observation to its F_{t-1} shift), so every model is cut to the shared dates.
# Comparing models over different test periods would not be a fair comparison.
common_idx = None
for df in model_data.values():
    common_idx = df.index if common_idx is None else common_idx.intersection(df.index)
common_idx = common_idx.sort_values()
 
assert len(common_idx) > 0, "No overlapping out-of-sample dates across the four models."
print(f"Common backtest period: {len(common_idx)} days "
      f"({common_idx.min().date()} to {common_idx.max().date()})")
 
aligned = {name: df.loc[common_idx] for name, df in model_data.items()}
 
# All four files carry the same actual P&L - confirm, then use one copy.
pnl = aligned['Historical']['Daily_PnL']
for name, df in aligned.items():
    assert np.allclose(df['Daily_PnL'].values, pnl.values), f"Daily_PnL differs in the {name} file."
n_obs = len(common_idx)

Loading model outputs from Phases 4-7...
  Historical  756 out-of-sample days  (2023-07-20 to 2026-07-24)
  EWMA        755 out-of-sample days  (2023-07-21 to 2026-07-24)
  GARCH       756 out-of-sample days  (2023-07-20 to 2026-07-24)
  GJR-GARCH   756 out-of-sample days  (2023-07-20 to 2026-07-24)
Common backtest period: 755 days (2023-07-21 to 2026-07-24)


In [11]:
# Step 5: Backtesting every model x window x confidence combination

In [12]:
summary_rows = []
timeline = {}
exception_frames = []
 
for model, (_, prefix) in MODEL_FILES.items():
    for w in windows:
        for cl in confidence_levels:
            col = f"{prefix}{w}d_{cl * 100}%"
            var_series = aligned[model][col]
            assert var_series.notna().all(), f"Unexpected NaN in {model} / {col}"
 
            # Exception = actual loss exceeds VaR  (P&L worse than -VaR), strictly greater
            hits = (pnl.values < -var_series.values)
            x = int(hits.sum())
            p = 1 - cl
            expected = n_obs * p
 
            lr_pof, p_pof = kupiec_pof(x, n_obs, p)
            lr_ind, p_ind = christoffersen_independence(hits)
            lr_cc, p_cc = conditional_coverage(lr_pof, lr_ind)
 
            # Basel traffic light: officially defined for 99% VaR over 250 days
            is_basel = bool(np.isclose(cl, 0.99)) and n_obs >= BASEL_WINDOW
            if is_basel:
                last250 = int(hits[-BASEL_WINDOW:].sum())
                rolling = pd.Series(hits.astype(int)).rolling(BASEL_WINDOW).sum().dropna().values
                zones = np.select([rolling <= 4, rolling <= 9], ['Green', 'Yellow'], default='Red')
                pct_green = float(np.mean(zones == 'Green'))
                pct_yellow = float(np.mean(zones == 'Yellow'))
                pct_red = float(np.mean(zones == 'Red'))
                zone_last = basel_zone(last250)
            else:
                last250, zone_last = np.nan, 'N/A'
                pct_green = pct_yellow = pct_red = np.nan
 
            kupiec_ok = p_pof >= SIGNIFICANCE
            cc_ok = True if np.isnan(p_cc) else (p_cc >= SIGNIFICANCE)
            result = 'PASS' if (kupiec_ok and cc_ok) else 'FAIL'
 
            # Which individual test is the weak spot? A model can PASS the combined
            # conditional-coverage test while one component is already rejecting.
            weak = []
            if p_pof < SIGNIFICANCE:
                weak.append('Kupiec')
            if not np.isnan(p_ind) and p_ind < SIGNIFICANCE:
                weak.append('Independence')
            weak_spot = ', '.join(weak) if weak else 'None'
 
            summary_rows.append({
                'Model': model, 'Window': w, 'Confidence': cl, 'Obs': n_obs,
                'Exceptions': x, 'Expected_Exceptions': round(expected, 2),
                'Exception_Ratio': round(x / expected, 3),
                'Kupiec_LR': lr_pof, 'Kupiec_p': p_pof,
                'Indep_LR': lr_ind, 'Indep_p': p_ind,
                'CC_LR': lr_cc, 'CC_p': p_cc,
                'Basel_Last250_Exceptions': last250, 'Basel_Last250_Zone': zone_last,
                'Basel_Pct_Windows_Green': pct_green,
                'Basel_Pct_Windows_Yellow': pct_yellow,
                'Basel_Pct_Windows_Red': pct_red,
                'Low_Power_Flag': x < 5,
                'Weak_Spot': weak_spot,
                'Result': result,
            })
 
            timeline[f"{model}|{w}d|{cl * 100}%"] = hits.astype(int)
 
            if x > 0:
                idx = np.where(hits)[0]
                exception_frames.append(pd.DataFrame({
                    'Date': common_idx[idx], 'Model': model, 'Window': w, 'Confidence': cl,
                    'Daily_PnL': pnl.values[idx],
                    'VaR_1d_$': var_series.values[idx],
                    'Excess_Loss_$': -pnl.values[idx] - var_series.values[idx],
                }))
 
summary = pd.DataFrame(summary_rows)
timeline_df = pd.DataFrame(timeline, index=common_idx)
exceptions_table = (pd.concat(exception_frames, ignore_index=True)
                    if exception_frames else pd.DataFrame())

In [13]:
# Step 6: Validation of the backtest

In [14]:
# (1) The exceptions table must agree with the summary counts
assert len(exceptions_table) == int(summary['Exceptions'].sum()), "Exceptions table/summary mismatch"
 
# (2) For one model and one window, a higher confidence level means a larger VaR,
#     so the exception count can never go UP as confidence rises.
for model in MODEL_FILES:
    for w in windows:
        sub = summary[(summary.Model == model) & (summary.Window == w)].sort_values('Confidence')
        counts = sub['Exceptions'].tolist()
        assert counts == sorted(counts, reverse=True), f"Exceptions rose with confidence for {model} {w}d"
 
print("Backtest machinery validated: exceptions table matches summary; "
      "exception counts fall as confidence rises for every model/window.")

Backtest machinery validated: exceptions table matches summary; exception counts fall as confidence rises for every model/window.


In [15]:
# Step 7: Report

In [16]:
show = summary.copy()
show['Conf'] = (show['Confidence'] * 100).map(lambda v: f"{v:g}%")
for c in ['Kupiec_p', 'Indep_p', 'CC_p']:
    show[c] = show[c].round(4)
print(f"\nBacktest results over {n_obs} common out-of-sample days "
      f"(PASS = Kupiec p and Conditional Coverage p both >= {SIGNIFICANCE}):\n")
print(show[['Model', 'Window', 'Conf', 'Exceptions', 'Expected_Exceptions', 'Exception_Ratio',
            'Kupiec_p', 'Indep_p', 'CC_p', 'Basel_Last250_Zone', 'Weak_Spot', 'Result']].to_string(index=False))
 
n_low = int(summary['Low_Power_Flag'].sum())
if n_low:
    print(f"\nNote: {n_low} of {len(summary)} combinations have fewer than 5 exceptions - "
          f"the tests have little power there, so a PASS is weak evidence.")
 
print("\nPass count by model:")
print(summary.groupby('Model')['Result'].apply(lambda s: f"{(s == 'PASS').sum()} of {len(s)} pass").to_string())


Backtest results over 755 common out-of-sample days (PASS = Kupiec p and Conditional Coverage p both >= 0.05):

     Model  Window  Conf  Exceptions  Expected_Exceptions  Exception_Ratio  Kupiec_p  Indep_p   CC_p Basel_Last250_Zone    Weak_Spot Result
Historical     250   95%          37                37.75            0.980    0.9000   0.0366 0.1116                N/A Independence   PASS
Historical     250   99%          13                 7.55            1.722    0.0706   0.0167 0.0111              Green Independence   FAIL
Historical     250 99.5%           7                 3.78            1.854    0.1372   0.0009 0.0013                N/A Independence   FAIL
Historical     500   95%          35                37.75            0.927    0.6422   0.0936 0.2200                N/A         None   PASS
Historical     500   99%           6                 7.55            0.795    0.5565   0.0338 0.0885              Green Independence   PASS
Historical     500 99.5%           5           

In [17]:
# SSaving the results

In [18]:
summary.to_csv('backtest_summary.csv', index=False)
timeline_df.to_csv('backtest_exceptions_timeline.csv')
exceptions_table.to_csv('backtest_exceptions_table.csv', index=False)
print("\nSaved: backtest_summary.csv, backtest_exceptions_timeline.csv, backtest_exceptions_table.csv")


Saved: backtest_summary.csv, backtest_exceptions_timeline.csv, backtest_exceptions_table.csv
